<!--
Copyright (c) Meta Platforms, Inc. and affiliates.
All rights reserved.

This source code is licensed under the CC-BY-NC-4.0 license found in the
LICENSE file in the root directory of this source tree.
-->

In [ ]:
import os, sys
from pathlib import Path
REPO = Path(os.environ.get('LONG_AUDIO_REPO', Path.cwd())).resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

# %% Imports
import json
import os
from collections import Counter
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_theme()  # Applies seaborn's default aesthetic

# %% Configuration
GT_PATH = REPO / "datasets/SINS/manifest.json"


def _arms(parent: Path, prefix: str = ""):
    """List subdirs of `parent`, stripping `prefix` from each name."""
    if not parent.is_dir():
        return []
    out = []
    for d in sorted(os.listdir(parent)):
        full = parent / d
        if not full.is_dir():
            continue
        if prefix and not d.startswith(prefix):
            continue
        out.append(d[len(prefix) :] if prefix else d)
    return out


ARMS = {
    "Casc. Gemini 3.5": str(REPO / "runs/desc-v4-20260715T160821Z/cascB_gemini/cascA_af3-hf/sins"),
    "Casc. GPT 5.6": str(REPO / "runs/desc-v4-20260715T160821Z/cascB_luna/cascA_af3-hf/sins"),
    "Casc. Gemma 4": str(REPO / "runs/desc-v4-20260715T160821Z/cascB_gemma4-31b/cascA_af3-hf/sins"),
    "Casc. Gemma 4 + SFT": str(REPO / "runs/desc-v4-20260715T160821Z/cascB_gemma4-31b_ft/cascA_af3-hf/sins"),
    "E2E Gemini 3.5": str(REPO / "runs/desc-v4-20260715T160821Z/e2e_gemini/sins"),
    # "E2E Gemini 3.5 + Think": str(REPO / "runs/thinking-20260728T195853Z/e2e_gemini_thinking4096/sins"),
    "E2E Qwen 3-o": str(REPO / "runs/desc-v4-20260715T160821Z/e2e_qwen3-omni/sins"),
    "E2E Qwen 3-o + SFT": str(REPO / "runs/desc-v4-20260715T160821Z/e2e_qwen3-omni_ft/sins"),
    # "No thinking": str(REPO / "runs/desc-v4-20260715T160821Z/cascB_luna/cascA_af3-hf/sins"),
    # "Low": str(REPO / "runs/thinking-20260728T195853Z/cascB_gemini_thinking1024/cascA_af3-hf/sins"),
    # "High": str(REPO / "runs/thinking-20260805T011616Z/cascB_luna_reasoning_high/cascA_af3-hf/sins"),
    # "No thinking": str(REPO / "runs/desc-v4-20260715T160821Z/e2e_gemini/sins"),
    # "1024": str(REPO / "runs/thinking-20260728T195853Z/e2e_gemini_thinking1024/sins"),
    # "4096": str(REPO / "runs/thinking-20260728T195853Z/e2e_gemini_thinking4096/sins"),
    # "E2E Qwen 3-o": str(REPO / "runs/desc-v4-20260715T160821Z/e2e_qwen3-omni/sins"),
    # "thinking 1024": str(REPO / "runs/thinking-20260728T195853Z/e2e_qwen3-omni_thinking1024/sins"),
    # "thinking 2048": str(REPO / "runs/thinking-20260728T195853Z/e2e_qwen3-omni_thinking2048/sins"),
    # "E2E Qwen 3-o + SFT": str(REPO / "runs/desc-v4-20260715T160821Z/e2e_qwen3-omni_ft/sins"),
    # "thinking 1024": str(REPO / "runs/thinking-20260728T195853Z/e2e_qwen3-omni_thinking1024/sins"),
    # "thinking 2048": str(REPO / "runs/thinking-20260728T195853Z/e2e_qwen3-omni_thinking2048/sins"),
}


# %% Load predictions
def load_predictions(model_dir: Path) -> list[dict]:
    chunks = sorted(model_dir.glob("chunk_*.json"))
    all_segs = []
    for cf in chunks:
        rec = json.loads(cf.read_text())
        all_segs.extend(rec.get("segments_abs", []))
    return all_segs


model_preds: dict[str, list[dict]] = {}
for name, path in ARMS.items():
    model_preds[name] = load_predictions(Path(path))

for name, preds in model_preds.items():
    print(f"  {name}: {len(preds)} segments")

# %% Load ground truth
gt_events = None
if GT_PATH.exists():
    manifest = json.loads(GT_PATH.read_text())
    gt_events = [
        {"label": a["event"], "start": float(a["start"]), "end": float(a["end"])}
        for a in manifest["annotations"]
    ]
    print(f"  GT: {len(gt_events)} segments")

# %% Build color map
all_labels = set()
for preds in model_preds.values():
    all_labels.update(e["label"] for e in preds)
if gt_events:
    all_labels.update(e["label"] for e in gt_events)
all_labels = sorted(all_labels)

# cmap = plt.cm.tab20(np.linspace(0, 1, max(1, len(all_labels))))
cmap = [
    "#DDCC77",
    "#CC6677",
    "#882255",
    "#AA4499",
    "#0072B2",
    "#332288",
    "#88CCEE",
    "#44AA99",
    "#117733",
    "#999933",
    "#CC79A7",
    "#661100",
    "#6699CC",
    "#888888",
    "#333333",
    "#E69F00",
    "#56B4E9",
    "#009E73",
    "#F0E442",
    "#D55E00",
]
label_colors = {label: cmap[i] for i, label in enumerate(all_labels)}

print(f"Labels ({len(all_labels)}): {all_labels}")

# %% [markdown]
# ## Timeline Strip Plot


# %%
def plot_strips(
    models,
    model_preds,
    gt_events=None,
    start_h=0,
    end_h=6,
    title="Segmentation Timeline",
):
    start_s = start_h * 3600
    end_s = end_h * 3600

    rows = []
    row_labels = []
    if gt_events:
        rows.append(gt_events)
        row_labels.append("Ground Truth")
    for model in models:
        rows.append(model_preds[model])
        row_labels.append(model)

    fig, axes = plt.subplots(
        len(rows),
        1,
        figsize=(12, 0.25 * len(rows)),
        sharex=True,
        constrained_layout=True,
    )
    if len(rows) == 1:
        axes = [axes]

    for ax, events, label in zip(axes, rows, row_labels):
        for e in events:
            s = float(e["start"])
            en = float(e["end"])
            if en <= start_s or s >= end_s:
                continue
            s = max(s, start_s)
            en = min(en, end_s)
            color = label_colors.get(e["label"], (0.5, 0.5, 0.5, 1.0))
            ax.axvspan(s / 3600, en / 3600, color=color, linewidth=0)
        ax.set_xlim(start_h, end_h)
        ax.set_yticks([])
        ax.set_ylabel(label, fontsize=8, rotation=0, ha="right", va="center")

    axes[-1].set_xlabel("Time (hours)", fontsize=10, labelpad=-5)
    axes[-1].tick_params(axis='x', labelsize=10, pad=-4)
    if title:
        axes[0].set_title(title, fontsize=10)

    patches = [mpatches.Patch(color=label_colors[l], label=l) for l in all_labels]
    fig.legend(
        handles=patches,
        loc="lower center",
        ncol=min(9, len(patches)),
        bbox_to_anchor=(0.57, -0.19),
        fontsize=8,
    )
    plt.savefig("/tmp/viz.pdf", bbox_inches="tight")
    plt.show()

In [ ]:
plot_strips(list(model_preds), model_preds, gt_events, start_h=0, end_h=24, title=None)

In [ ]:
# import matplotlib
# import os, shutil

# # Find and remove the font cache
# cache_dir = matplotlib.get_cachedir()
# print("Cache dir:", cache_dir)
# for f in os.listdir(cache_dir):
#     if f.startswith("fontlist") or f.endswith(".cache"):
#         os.remove(os.path.join(cache_dir, f))
#         print("Removed", f)